<a href="https://colab.research.google.com/github/Mmbsaksd/transformers/blob/main/byte_level_vs_normal_bpe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Byte-Level BPE vs. Character-Level (Normal) BPE

### A hands-on, from-scratch comparison

**Audience:** students who already know *what* tokenization is and have seen
Byte Pair Encoding (BPE) described at a high level ("merge the most frequent
adjacent pair, repeat").

**Goal of this notebook:** answer one very specific question —

> *When people say GPT-2/GPT-3/GPT-4 use "byte-level BPE", what exactly is
> different from the "normal" BPE algorithm described in the original 2016
> paper (Sennrich et al., *Neural Machine Translation of Rare Words with
> Subword Units*)?*

The surprising answer: **the merging algorithm itself does not change at
all.** Only one thing changes — *what counts as an atomic symbol before any
merging starts.* Everything else (count pairs → merge the most frequent pair
→ repeat) is identical.

We will:
1. Implement the shared BPE engine **once**.
2. Run it in **character mode** (the "normal"/classic version).
3. Run the *exact same engine* in **byte mode**.
4. Break the character version with an unseen symbol, and show the byte
   version can never be broken that way.
5. Confirm real production tokenizers (`tiktoken`, used by GPT-2/3.5/4) are
   byte-level, and see it with our own eyes on emojis, Hindi text, and math
   symbols.


## 1. The BPE engine — shared by both versions

Recall the algorithm, regardless of what a "symbol" is:

1. Start with a sequence of atomic symbols for every word in the training
   corpus.
2. Count how often every **adjacent pair** of symbols occurs, across the
   whole corpus.
3. Take the **single most frequent pair** and merge it into one new symbol.
   Add that new symbol to the vocabulary.
4. Repeat steps 2–3 until you hit your target vocabulary size (or no pair
   occurs more than once).

Nothing here mentions "character" or "byte". The engine below is written to
be agnostic to that choice — it just operates on a list of **symbols**,
whatever those symbols happen to be. We'll feed it two different kinds of
atoms later.


In [1]:
from collections import Counter

def get_pair_counts(corpus_symbols):
    '''
    corpus_symbols: list of words, where each word is itself a list of symbols.
    e.g. [['l','o','w'], ['l','o','w','e','r']]

    Returns a Counter mapping (symbol_a, symbol_b) -> number of times that
    adjacent pair occurs, summed across every word in the corpus.
    '''
    pairs = Counter()
    for word in corpus_symbols:
        for a, b in zip(word[:-1], word[1:]):
            pairs[(a, b)] += 1
    return pairs


def merge_pair_in_corpus(corpus_symbols, pair_to_merge):
    '''
    Walks every word and replaces every occurrence of pair_to_merge=(a, b)
    with the single merged symbol (a + b), stitched together.
    '''
    a, b = pair_to_merge
    merged_symbol = a + b
    new_corpus = []
    for word in corpus_symbols:
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and word[i] == a and word[i + 1] == b:
                new_word.append(merged_symbol)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_corpus.append(new_word)
    return new_corpus


def train_bpe(corpus_symbols, num_merges, verbose=True):
    '''
    The ENGINE. Identical for character-level and byte-level BPE.
    Returns:
        corpus_symbols  -> the corpus, fully merged
        merges          -> ordered list of merge rules learned, e.g. [(('l','o'), 'lo'), ...]
    '''
    merges = []
    for step in range(1, num_merges + 1):
        pair_counts = get_pair_counts(corpus_symbols)
        if not pair_counts:
            if verbose:
                print(f"Step {step}: no more pairs to merge. Stopping early.")
            break

        best_pair, count = pair_counts.most_common(1)[0]
        corpus_symbols = merge_pair_in_corpus(corpus_symbols, best_pair)
        merges.append((best_pair, best_pair[0] + best_pair[1]))

        if verbose:
            print(f"Step {step:>2}: merged {best_pair} -> {best_pair[0]+best_pair[1]!r}  "
                  f"(seen {count} times)")

    return corpus_symbols, merges

print("BPE engine defined. This exact code will be reused for BOTH experiments below.")


BPE engine defined. This exact code will be reused for BOTH experiments below.


## 2. Experiment A — "Normal" / classic character-level BPE

Here, the **atomic symbol = one Unicode character**. Before any merging, the
vocabulary is: *the set of unique characters that appear in the training
corpus* — nothing more, nothing less.

Toy corpus (word, frequency) — a classic tokenization teaching example:


In [7]:
toy_corpus = {
    "low":     5,
    "lower":   2,
    "newest":  6,
    "widest":  3,
}

# Expand frequency into repeated words, and split each word into CHARACTERS.
# This is the "normal" BPE starting point: atoms = characters.
char_corpus = []
for word, freq in toy_corpus.items():
    char_corpus.extend([list(word)] * freq)

for word in sorted(set(tuple(w) for w in char_corpus)):
    print(" ", word)


initial_char_vocab = sorted(set(ch for word in char_corpus for ch in word))
print(f"\nInitial (base) vocabulary size = {len(initial_char_vocab)} characters")
print("Initial vocabulary:", initial_char_vocab)

  ('l', 'o', 'w')
  ('l', 'o', 'w', 'e', 'r')
  ('n', 'e', 'w', 'e', 's', 't')
  ('w', 'i', 'd', 'e', 's', 't')

Initial (base) vocabulary size = 10 characters
Initial vocabulary: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']


In [9]:
print("="*70)
print("TRAINING CHARACTER-LEVEL BPE")
print("="*70)

final_char_corpus, char_merges = train_bpe(char_corpus, num_merges=8)

print("\nFinal vocabulary after merges:")
final_char_vocab = set(initial_char_vocab)
for _, merged in char_merges:
    final_char_vocab.add(merged)
print(sorted(final_char_vocab, key=len))

print(f"\nTotal vocabulary size now: {len(final_char_vocab)} "
      f"({len(initial_char_vocab)} base characters + {len(char_merges)} learned merges)")


TRAINING CHARACTER-LEVEL BPE
Step  1: merged ('e', 's') -> 'es'  (seen 9 times)
Step  2: merged ('es', 't') -> 'est'  (seen 9 times)
Step  3: merged ('l', 'o') -> 'lo'  (seen 7 times)
Step  4: merged ('lo', 'w') -> 'low'  (seen 7 times)
Step  5: merged ('n', 'e') -> 'ne'  (seen 6 times)
Step  6: merged ('ne', 'w') -> 'new'  (seen 6 times)
Step  7: merged ('new', 'est') -> 'newest'  (seen 6 times)
Step  8: merged ('w', 'i') -> 'wi'  (seen 3 times)

Final vocabulary after merges:
['o', 't', 'w', 'n', 'i', 'r', 'l', 'e', 's', 'd', 'lo', 'wi', 'es', 'ne', 'low', 'new', 'est', 'newest']

Total vocabulary size now: 18 (10 base characters + 8 learned merges)


### 2.1 The weak point of character-level BPE

The base vocabulary was built **only from characters seen during training**.
What happens if, at inference time, someone hands the tokenizer a character
it has *never* seen before — say, an emoji, or a Chinese character, if the
training data was pure English?

There is no rule to fall back on. Classic implementations solve this with a
special `<UNK>` (unknown) token, which **throws away information** — the
model has no idea what that character actually was.

Let's simulate that failure honestly:


In [11]:
def encode_char_bpe(word, merges, base_vocab):
    '''
    Encodes a single word using the character-level BPE we just trained.
    Any character NOT in base_vocab cannot even be represented as a starting
    symbol, so we mark it explicitly as unknown.
    '''
    symbols = list(word)

    # Check every character against what the tokenizer has ever seen.
    unknown_chars = [ch for ch in symbols if ch not in base_vocab]
    if unknown_chars:
        print(f"  !! Character(s) {unknown_chars} were NEVER seen during training.")
        print(f"  !! Classic BPE has no way to represent them -> falls back to <UNK>.")
        return None

    # Apply learned merges in the order they were learned.
    for (a, b), merged in merges:
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols


print("Encoding a normal, in-vocabulary word: 'lowest'")
result = encode_char_bpe("lowest", char_merges, initial_char_vocab)
print("  Tokens:", result)

print("\nEncoding a word containing an UNSEEN character: 'low🙂'")
result = encode_char_bpe("low🙂", char_merges, initial_char_vocab)
print("  Result:", result)


Encoding a normal, in-vocabulary word: 'lowest'
  Tokens: ['low', 'est']

Encoding a word containing an UNSEEN character: 'low🙂'
  !! Character(s) ['🙂'] were NEVER seen during training.
  !! Classic BPE has no way to represent them -> falls back to <UNK>.
  Result: None


**This is the core limitation.** Character-level BPE's base alphabet is
*whatever characters happened to appear in the training set*. Unicode has
over 150,000 possible characters (emoji, every world script, symbols, ...).
No training corpus contains all of them, so `<UNK>` is always a risk in
production — and every time it fires, information is silently destroyed.

## 3. Experiment B — Byte-level BPE

Now we make **exactly one change** to the recipe: instead of splitting words
into Unicode *characters*, we first encode each word as raw **UTF-8 bytes**,
and split into *those*.

Why does this matter? Because **every possible piece of text in every
language, emoji included, is made of some sequence of bytes from 0–255.**
That is what UTF-8 guarantees. So if our atomic alphabet is "the 256 possible
byte values", the alphabet is *complete by construction* — there is no such
thing as a byte the tokenizer has never heard of, because we don't even need
to have seen it: 0–255 is the entire universe of possibilities, decided
upfront, not learned from data.

Everything else — `get_pair_counts`, `merge_pair_in_corpus`, `train_bpe` — is
the **exact same code** we already wrote above.


In [ ]:
def word_to_bytes_symbols(word):
    '''
    THIS is the one-line algorithmic difference between byte-level and
    character-level BPE. Instead of list(word) [-> characters], we do:
    '''
    utf8_bytes = word.encode("utf-8")          # str -> raw bytes
    return [bytes([b]) for b in utf8_bytes]    # split into single-byte symbols

